# {{PROJECT_TITLE}}
## Modelling Human Activity States with a From-Scratch Hidden Markov Model
**Author:** {{YOUR_NAME}}

**Use case:** {{USE_CASE}}

| Section | Rubric criterion |
|---|---|
| 1–3 Data, harmonization, visualization, windowing | Data Collection & Quality (10) |
| 4–5 Features & normalization | Feature Extraction (10) |
| 6–8 HMM components, implementation, Baum–Welch | Implementation (15) |
| 9–10 Visualizing A & B, evaluation on unseen data | Evaluation (10) |

> Run with **Kernel → Restart & Run All**. All figures are saved to `../reports/figures/`.

## 0. Configuration

In [ ]:
import sys, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))          # so `src` is importable
%load_ext autoreload
%autoreload 2

DATA = Path("../data"); FIG = Path("../reports/figures"); FIG.mkdir(parents=True, exist_ok=True)
STATES = ["standing", "walking", "jumping", "still"]
TARGET_FS  = 50.0   # Hz
WINDOW_SEC = 2.0    # s -> 100 samples
OVERLAP    = 0.5    # 1 s hop
TRIM_SEC   = 1.0    # s trimmed from each end
BW_TOL     = 1e-4   # relative change in log-likelihood
SEED = 42; rng = np.random.default_rng(SEED)

## 1. Data loading & cleaning
Raw exports → merged accel+gyro → cleaned CSVs. *(Build `src/preprocessing.py` with Prompt 2.)*

**Write here:** collection protocol in 3 bullets, number of files per activity, how trimming/cleaning was done.

In [ ]:
# TODO (Prompt 2): from src.preprocessing import build_clean_dataset
# summary = build_clean_dataset(DATA/"raw", DATA, TARGET_FS)
# summary.groupby("activity").agg(files=("file","count"), seconds=("duration_s","sum"))

## 2. Sampling-rate harmonization
**Rubric:** list the measured rate of every device and explain how rates were harmonized.

| Device | Used for | Accel Hz (measured) | Gyro Hz (measured) |
|---|---|---|---|
| {{PHONE}} | training + test | {{FS_ACC}} | {{FS_GYR}} |

**Justification:** {{accel and gyro run on separate clocks at slightly different rates and timestamps jitter -> gyro interpolated onto the accel clock, then everything linearly resampled to a uniform 50 Hz grid; 50 Hz <= native rate (no invented data), Nyquist 25 Hz > human motion (< 20 Hz); low-pass first if native >= 100 Hz (aliasing)}}


In [ ]:
# TODO: print measured sampling rates per device (summary.groupby("device")[["native_fs_acc","native_fs_gyro"]].median())

## 3. Raw-signal visualization *(goes in report as Figure 1)*

In [ ]:
# TODO (Prompt 2): 2x4 grid — accel axes + |a| (top), gyro axes (bottom), one example per activity
# plt.savefig(FIG/"raw_signals.png", dpi=200, bbox_inches="tight")

### 3.1 Windowing
**Window = {{WINDOW_SEC}} s = {{WIN_SAMPLES}} samples at {{TARGET_FS}} Hz, {{OVERLAP}} overlap.**
Justification: ≥ 3 walking/jumping cycles (~1.5–2.5 Hz) per window; FFT resolution = fs/N = {{FREQ_RES}} Hz; 1 s hop = HMM time step.

## 4. Feature extraction *(Prompt 3)*
| # | Feature | Domain | Separates | Why |
|---|---|---|---|---|
| 1 | acc_mag_mean | Time | | |
| … | dom_freq | FFT | moving vs not moving (walking and jumping cadences overlap at ~1.5–2.5 Hz, so do not claim it separates them) | |

In [ ]:
# TODO (Prompt 3): from src.features import make_windows, extract_features, build_feature_matrix, ZScoreNormalizer
# X_train, y_train, groups_train, feature_names = build_feature_matrix(train_files)
# print(X_train.shape, np.bincount(y_train))

## 5. Normalization
Z-score with μ, σ **fitted on training windows only** — justification: {{units differ; Gaussian emissions scale-sensitive; robust to jump-impact outliers vs min-max; avoids leakage}}.

In [ ]:
# TODO: norm = ZScoreNormalizer().fit(X_train); Xtr = norm.transform(X_train)
# TODO: violin/box plots of 4 key features by activity -> FIG/"feature_distributions.png"

## 6. HMM components
| Element | Definition |
|---|---|
| Hidden states Z | standing, walking, jumping, still |
| Observations X | {{N_FEATURES}}-dim Z-scored feature vector per window |
| Transitions A | 4×4, init: {{A_INIT_METHOD}}, refined by Baum–Welch |
| Emissions B | diagonal Gaussian per state, variance floor {{VAR_FLOOR}} |
| Initial π | {{PI}} |

### Algorithm choice (brief — tie to metrics)
- **Forward–backward:** posteriors γ and log P(X); needed for the E-step and for confidence.
- **Baum–Welch (EM):** learns π, A, B by maximizing likelihood; stop when Δ log-lik < ε.
- **Viterbi:** most probable *sequence* of states → the labels we score with sensitivity, specificity, accuracy; uses A to smooth isolated misclassified windows.

## 7. Implementation *(src/hmm.py, Prompt 4 — verified by `pytest tests/`)*

In [ ]:
from src.hmm import GaussianHMM
# Run the correctness tests from inside the notebook too:
# !cd .. && python -m pytest tests/ -q

## 8. Training: supervised init → Baum–Welch *(Prompt 5)*

In [ ]:
# TODO: build realistic training sequences (concatenate clips along plausible transitions)
# model = GaussianHMM(n_states=4, n_features=Xtr.shape[1], random_state=SEED)
# model.fit_supervised(train_seqs, train_labels)
# model.fit(train_seqs, tol=BW_TOL, max_iter=500)
# print("converged:", model.converged_, "iterations:", model.n_iter_)
# plot model.history_ -> FIG/"bw_convergence.png"
# check label permutation (best of 24 permutations)

## 9. Visualizing the learned model

In [ ]:
# TODO: annotated heatmap of model.transmat_ -> FIG/"transition_matrix.png"
# TODO: heatmap of model.means_ (states x features) + Gaussian curves of top-3 features -> FIG/"emissions.png"
# print("π =", np.round(model.startprob_, 3))

**Interpretation of A:** {{diagonal persistence; near-zero still→jumping because the phone must be picked up first; …}}

## 10. Evaluation on unseen data *(Prompt 6)*
**How the unseen data was obtained:** {{TEST_SOURCE}}

In [ ]:
# TODO: for each test file: window -> features -> norm.transform (TRAIN stats) -> model.viterbi
# TODO: baseline = argmax of model.log_emission per window
# TODO: confusion matrix (counts + row %) -> FIG/"confusion_matrix.png"
# TODO: decoded-sequence plot (|a| + True / Viterbi / Baseline stripes) -> FIG/"decoded_test_01.png", "decoded_test_02.png"

In [ ]:
def per_state_metrics(cm, names):
    """Sensitivity = TP/(TP+FN), Specificity = TN/(TN+FP) per state, from a KxK confusion matrix (rows = true)."""
    total = cm.sum(); rows = []
    for k, n in enumerate(names):
        tp = cm[k, k]; fn = cm[k].sum() - tp; fp = cm[:, k].sum() - tp; tn = total - tp - fn - fp
        rows.append({"State (Activity)": n, "Number of Samples": int(cm[k].sum()),
                     "Sensitivity": tp / (tp + fn) if tp + fn else np.nan,
                     "Specificity": tn / (tn + fp) if tn + fp else np.nan})
    df = pd.DataFrame(rows); df["Overall Accuracy"] = np.trace(cm) / total
    return df.round(3)
# display(per_state_metrics(cm, STATES))   # copy into report Table 5

## 11. Analysis & reflection (draft here, polish in the report)
1. **Easiest / hardest activities:** {{...}}
2. **Transitions vs realistic behaviour:** {{...}}
3. **Sensor noise & sampling rate effects:** {{...}}
4. **Improvements:** {{...}}